# 12장 파인튜닝과 나만의 챗봇

- 학습하는 셀은 GPU에서 실행합니다(런타임 → 런타임 유형 변경 → T4 GPU). GPU가 없으면 학습 셀을 건너뛰세요. 다음 셀에서 미리 학습시켜 둔 파일을 내려받아 이어 갑니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

데이터: [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, CC BY 4.0) · 모델: [JosephLee/modu-llm](https://huggingface.co/JosephLee/modu-llm) (CC BY 4.0)

## 실습 12-1 준비: 사전학습 모델 불러오기

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import torch
from modu_llm import Vocab, END, encode_texts, generate, format_pair
from modu_llm import load_checkpoint, load_sft_texts, get_file, train, mount_drive
from modu_llm import get_device

device = get_device(need_gpu=True)
vocab = Vocab.build()
folder = mount_drive()
pre_model, _, pre_step, _ = load_checkpoint(get_file("pretrain.pt"), "cpu")
print(f"사전학습 모델: {pre_step}번 학습")

## 실습 12-2 사전학습 모델은 왜 요청을 못 따를까

**먼저 예상해 보세요.** 사전학습 모델은 토끼 보리가 나오는 이야기를 쓸까요?

In [ ]:
question = format_pair("숲속에서 토끼 보리가 나오는 이야기를 들려줘")
print(repr(question))
torch.manual_seed(0)
answer = generate(pre_model, vocab, question, max_new_tokens=150)
print(question + answer)

## 실습 12-3 요청-이야기 데이터

In [ ]:
texts = load_sft_texts("train")
val_texts = load_sft_texts("validation")
print(f"학습용 {len(texts):,}쌍 | 검증용 {len(val_texts):,}쌍")
print(texts[0] + END)

## 실습 12-4 사전학습 모델에서 이어 학습하기

In [ ]:
train_data = encode_texts(texts, vocab)
val_data = encode_texts(val_texts, vocab)
sft_path = f"{folder}/sft.pt"

model, optimizer, _, _ = load_checkpoint(get_file("pretrain.pt"), device, optimizer_lr=1e-4)
step, history = train(model, optimizer, train_data, val_data,
                      minutes=5, ckpt_path=sft_path, eval_every=200,
                      keep_best=True)
print(f"학습 끝: {step}번")

## 실습 12-5 같은 요청, 달라진 대답

**먼저 예상해 보세요.** 파인튜닝 모델은 요청에 나온 인물로 이야기를 시작할까요?

In [ ]:
model, _, best_step, _ = load_checkpoint(get_file("sft.pt"), "cpu")
print(f"파인튜닝 모델: {best_step}번")
for r in ["숲속에서 토끼 보리가 나오는 이야기를 들려줘", "강아지 뭉치가 나오는 이야기를 만들어 줘"]:
    print("요청:", r)
    for name, m in [("사전학습", pre_model), ("파인튜닝", model)]:
        torch.manual_seed(0)
        answer = generate(m, vocab, format_pair(r), max_new_tokens=120)
        print(f"  [{name}]", answer.replace("\n", " "))

## 실습 12-6 chat 함수

**먼저 예상해 보세요.** 요청에 "부엌"을 넣으면 이야기도 부엌에서 시작할까요?

In [ ]:
def chat(request, temperature=0.8):
    question = format_pair(request)                        # ① 요청을 형식에 담기
    story = generate(model, vocab, question, max_new_tokens=300,
                     temperature=temperature, top_k=20)    # ② 이야기 이어 쓰기
    return story.strip()                                   # ③ 앞뒤 빈칸 정리하기

torch.manual_seed(1)
print(chat("강아지 뭉치가 부엌에서 겪은 이야기를 들려줘"))

## 실습 12-7 대화 화면 띄우기

In [ ]:
%pip install -q gradio
from modu_llm import chat_app

demo = chat_app(chat)
demo.launch()